# ⚙️ Production Vector Stores: Qdrant, pgvector & Multi-Tenant Architecture

**My Objective:** I am migrating the retrieval layer from an embedded, single-process library (ChromaDB) to production client-server vector databases (Qdrant and PostgreSQL with `pgvector`)[cite: 5]. Enterprise deployments require network-accessible services that survive application crashes, scale horizontally, and enforce strict multi-tenant data isolation[cite: 5]. I am executing a zero-re-embed migration, verifying retrieval parity via `overlap@k`, auditing tenant data leaks, and modeling memory footprints across vector quantization tiers[cite: 5].

In [ ]:
import sys
import os
from pathlib import Path

# Secure, dynamic path resolution
WORK_REPO = Path(os.getenv("WORK_REPO", Path.cwd()))
sys.path.insert(0, str(WORK_REPO / "vectordbs") if (WORK_REPO / "vectordbs").exists() else str(WORK_REPO))

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv(usecwd=True))

import numpy as np
import embedding_utils as EMBEDDING
import vs_utils as VS

# Extract existing vectors and metadata from ChromaDB without re-embedding
emb = EMBEDDING.Embedder("bge-small")
rows = VS.chroma_rows(emb)
print(len(rows["ids"]), "rows; first id:", rows["ids"][0], "| metadata:", rows["metadatas"][0])

## 1. Zero-Re-Embed Migration to Qdrant
Because the underlying embedding model (`bge-small`) remains unchanged, re-embedding the corpus is unnecessary compute waste[cite: 5]. I am migrating the raw vectors and metadata directly into Qdrant, a dedicated Rust-based vector engine[cite: 5]. Since Qdrant requires numeric or UUID point identifiers, string IDs are deterministically mapped to UUIDs while preserving the original identifier inside the payload[cite: 5].

In [ ]:
client, where = VS.connect_qdrant("auto")      # Connects to Docker server if active, else in-process fallback
print("connected to:", where)
print("points written:", VS.migrate_to_qdrant(client, rows))

p = client.scroll("help_centre", limit=1, with_payload=True)[0][0]
print("one point -> id:", p.id, "| payload keys:", list(p.payload))

## 2. Relational Vector Storage (PostgreSQL + `pgvector`)
When an organization already operates PostgreSQL, `pgvector` eliminates the operational overhead of maintaining a separate vector infrastructure[cite: 5]. It introduces a native `vector(384)` column type, HNSW indexing, and the `<=>` cosine-distance operator directly inside standard SQL, enabling transactional integrity and relational joins alongside semantic search[cite: 5].

In [ ]:
print("CREATE EXTENSION IF NOT EXISTS vector;")
print(VS.PG_SCHEMA)
print(VS.PG_INDEX)
print(VS.PG_SEARCH)

In [ ]:
conn = VS.connect_pg()
print("rows written:", VS.migrate_to_pg(conn, rows))
print(conn.execute("SELECT id, category, updated, left(body, 50) FROM help_centre LIMIT 3").fetchall())

In [ ]:
q = emb.embed(["how long do refunds take"], kind="query")[0]
for score, row in VS.search_pg(conn, q, 3):
    print(f"{score:.3f}  {row['title']} > {row['heading']}")

print("\nwith a normal SQL WHERE:")
for score, row in VS.search_pg(conn, q, 3, category="returns"):
    print(f"{score:.3f}  {row['title']} > {row['heading']}  [{row['category']}]")

In [ ]:
# Standard relational aggregation alongside vector storage
print(conn.execute("SELECT category, count(*) FROM help_centre GROUP BY category ORDER BY 2 DESC").fetchall())

## 3. Cross-Database Parity Verification (`overlap@k`)
A database migration is only valid when retrieval behavior is empirically verified[cite: 5]. I am querying ChromaDB, Qdrant, and `pgvector` with an identical prompt and calculating `overlap@k` to confirm that all three engines return identical top-3 results[cite: 5].

In [ ]:
from vdb_utils import get_collection, open_db, search
question = "my parcel arrived broken"
qv = emb.embed([question], kind="query")[0]
chroma = [h["id"] for h in search(get_collection(open_db()), emb, question, 3)]
qdrant = [p["chroma_id"] for _, p in VS.search_qdrant(client, qv, 3)]
print("ChromaDB:", chroma)
print("Qdrant  :", qdrant, " overlap", VS.overlap_at_k(chroma, qdrant))
try:
    pgv = [p["chroma_id"] for _, p in VS.search_pg(conn, qv, 3)]
    print("pgvector:", pgv, " overlap", VS.overlap_at_k(chroma, pgv))
except NameError:
    print("pgvector: skipped (no Docker)")

## 4. Multi-Tenancy & Data Isolation Audit
Storing multiple customers' data in a shared collection creates severe cross-tenant leakage risks if queries are unfiltered[cite: 5]. Below, an unfiltered search leaks `CUST-1002`'s private notes into `CUST-1001`'s results[cite: 5]. Enforcing a mandatory `customer_id` payload filter—derived strictly from the authenticated login token rather than user input—eliminates the leak at the database layer[cite: 5].

In [ ]:
from qdrant_client import models

VS.migrate_notes(client, emb)
qv = emb.embed(["what did the customer ask about their refund"], kind="query")[0]

print("Asha's search WITHOUT a filter:")
for s, p in VS.search_qdrant(client, qv, 3, "customer_notes"):
    print(f"   [{p['customer_id']}] {p['text'][:70]}")

asha = models.Filter(must=[models.FieldCondition(key="customer_id", match=models.MatchValue(value="CUST-1001"))])
print("\nWITH customer_id = CUST-1001:")
for s, p in VS.search_qdrant(client, qv, 3, "customer_notes", query_filter=asha):
    print(f"   [{p['customer_id']}] {p['text'][:70]}")

## 5. Memory Economics & Vector Quantization
At scale, raw `float32` vectors consume massive RAM budgets (100M × 384-dim vectors require ~153.6 GB)[cite: 5]. Quantizing vectors to `int8` (4× reduction) or `binary` (32× reduction) drastically compresses infrastructure costs while preserving search throughput[cite: 5].

In [ ]:
for n in (1_000_000, 10_000_000, 100_000_000):
    print(f"{n:>12,} x 384:  float32 {n*384*4/1e9:7.1f} GB   int8 {n*384/1e9:6.1f} GB   binary {n*384/8/1e9:5.2f} GB")